In [ ]:
from langchain.messages import HumanMessage

print(HumanMessage(content='hello'))
print(HumanMessage('hello'))

# 识别解析图片内容

In [ ]:
import base64
import os
import requests
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_core.prompt_values import ChatPromptValue

# 加载配置文件
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL')



def get_content(url: str, timeout: int = 10) -> bytes:
    """从 URL 下载，返回原始字节"""
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()                 # 下载失败抛异常，别静默吞掉
    return resp.content

def encode_image(img_path,img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path,'rb') as img_file:
        return f'data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode('utf8')}'

def encode_image_from_bytes(img_bytes: bytes, img_type: str = 'jpeg') -> str:
    """把图片字节编码成 Data URI"""
    return f'data:image/{img_type};base64,{base64.b64encode(img_bytes).decode('utf-8')}'

# 获取图像base64编码字符串
base64_image = encode_image_from_bytes(get_content('https://gips2.baidu.com/it/u=1562350815,739640146&fm=3074&app=3074&f=JPEG?w=1922&h=2530&type=normal&func='))

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)
response = model.invoke(
    ChatPromptValue(messages=[
        HumanMessage(content_blocks =[
        {'type': 'text', 'text': '这张图里有什么？'},
        {
            'type': 'image_url',
            'image_url': {'url': base64_image},
        }
    ])
    ])
)
print(response.content)


# 提示词模板

In [ ]:
# 字符串拼接方式 实现提示词
topic = 'Python'
difficulty = '初学者'

prompt_str = f'你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。'

response = model.invoke(prompt_str)
print(f"AI 回复：{response.content}...\n")


In [ ]:
# PromptTemplate 提示词模板 实现提示词
from langchain_core.prompts import PromptTemplate
topic = 'Python'
difficulty = '中级开发者'

template = PromptTemplate.from_template(
    '你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。'
)

# 使用模板生成提示词
prompt = template.format(difficulty=difficulty,topic=topic)

response = model.invoke(prompt)
print(f"AI 回复：{response.content}...\n")


In [ ]:
# ChatPromptTemplate 提示词模板 实现提示词
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ('system','你是一个AI开发工程师. 你的名字是 {name}.'),
    ('human','{user_input}')
])
#调用format()方法，返回字符串
prompt = prompt_template.invoke({'name':'小kAI', 'user_input': '你能帮我做什么?'})
print(prompt)

In [ ]:
# 调用from_messages() 提示词模板 实现提示词
from langchain_core.prompts import ChatPromptTemplate

# 定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages([
    ('system','你是一个有帮助的AI机器人，你的名字是{name}。'),
    ('human','你好，最近怎么样？'),
    ('ai','我很好，谢谢！'),
    ('human','{user_input}')
])

# 格式化聊天提示词模版中的变量
prompt = chat_template.invoke({
    'name': '岚希儿',
    'user_input': '你叫什么名字？'
})

# 打印格式化后的聊天提示词模版内容
print(prompt)

In [ ]:
# 模板 invoke 方式调用
from langchain_core.prompts import ChatPromptTemplate

# 参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ('system','你是一个AI开发工程师. 你的名字是 {name}.'),
    ('human','你能开发哪些AI应用?'),
    ('ai','我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.'),
    ('human','{user_input}'),
])

prompt = prompt_template.invoke({
    'name': 'xiaoAI',
    'user_input': '你能帮我做什么？'
})
print(prompt)


In [ ]:
# 模板 format 方式调用
from langchain_core.prompts import ChatPromptTemplate

# 参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ('system','你是一个AI开发工程师. 你的名字是 {name}.'),
    ('human','你能开发哪些AI应用?'),
    ('ai','我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.'),
    ('human','{user_input}'),
])
# 调用format()方法，返回字符串
prompt = prompt_template.format(name='果冻AI', user_input='你能帮我做什么？')
print(prompt)

In [ ]:
# 模板 format_messages 方式调用
from langchain_core.prompts import ChatPromptTemplate

# 参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ('system','你是一个AI开发工程师. 你的名字是 {name}.'),
    ('human','你能开发哪些AI应用?'),
    ('ai','我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.'),
    ('human','{user_input}'),
])

# 调用format_messages()方法，返回消息列表
prompt = prompt_template.format_messages(name='miAI', user_input='你能帮我做什么？')
print(prompt)

In [ ]:
# 不带参数
from langchain_core.prompts import ChatPromptTemplate

# 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt_template = ChatPromptTemplate.from_messages([
    ('system','我是一个人工智能助手'),
])
nested_prompt_template_ = ChatPromptTemplate.from_messages([
     ('human','很高兴认识你')
])

prompt_template = ChatPromptTemplate.from_messages([
    nested_prompt_template, nested_prompt_template_
])

prompt_template.invoke({})

In [ ]:
# 带参数
from langchain_core.prompts import ChatPromptTemplate

# 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt_template = ChatPromptTemplate.from_messages([
    ('system','我是一个人工智能助手，我的名字叫{name}'),
])
nested_prompt_template_ = ChatPromptTemplate.from_messages([
     ('human','很高兴认识你，我的问题是{question}')
])

prompt_template = ChatPromptTemplate.from_messages([
    nested_prompt_template, nested_prompt_template_
])

prompt_template.invoke({
    'name': '小智',
    'question': '你为什么这么帅？'
})

In [ ]:
# 综合使用
from langchain_core.prompts import (ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate)
from langchain_core.messages import SystemMessage, HumanMessage

# 使用 BaseMessage（已实例化的消息）
system_msg = SystemMessage(content='你是一个AI工程师')
human_msg = HumanMessage(content='你好！')

# 使用 BaseMessagePromptTemplate
system_prompt = SystemMessagePromptTemplate.from_template('你是一个{role}.')
human_prompt = HumanMessagePromptTemplate.from_template('{user_input}')

# 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt = ChatPromptTemplate.from_messages([
    ('system','嵌套提示词')
])

prompt = ChatPromptTemplate.from_messages([
    system_msg, # MessageLike (BaseMessage)
    human_msg, # MessageLike (BaseMessage)
    system_prompt, # MessageLike (BaseMessagePromptTemplate)
    human_prompt, # MessageLike (BaseMessagePromptTemplate)
    nested_prompt, # MessageLike (BaseChatPromptTemplate)
])
prompt.invoke({
    'role':'人工智能专家',
    'user_input': '介绍一下大模型的应用场景'
})

In [ ]:
# 部分变量预填充：partial()
from langchain_core.prompts import ChatPromptTemplate

# 原始模板
template = ChatPromptTemplate.from_messages([
    ('system','你是{role}，目标用户是{audience}'),
    ('user', '{task}')
])

# 部分填充
customer_support_template = template.partial(
    role='客服专员',
    audience='普通用户'
)

# 现在只需要提供task
messages = customer_support_template.invoke({
    'task': '解释退款政策'
})
print(messages)

# 场景：为不同部门创建专用模板
base_template = ChatPromptTemplate.from_messages([
    ('system','你是{department}的{role}'),
    ('user','{task}')
])

# IT部门
it_template = base_template.partial(
    department= 'IT 部门',
    role= '技术支持'
)

# 销售部门
sales_template = base_template.partial(
    department= '销售部门',
    role= '销售顾问'
)
sales_template.invoke({
    'task': '为什么每年年底汽车会促销'
})

In [ ]:
#  消息占位符
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ('system','你是一个有用的AI助手'),
    ('placeholder','{conversation}')
])
prompt_value = template.invoke({
    'conversation': [
        ('human','你好'),
        ('ai','今天我能帮你做什么？'),
        ('human','你能给我做一个冰激凌吗？'),
        ('ai','抱歉，我没有这样的能力'),
    ]
})
print(prompt_value)

In [ ]:
# MessagesPlaceholder实例
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain.messages import HumanMessage
prompt_template = ChatPromptTemplate.from_messages([
    ('system','You are a helpful assistant'),
    MessagesPlaceholder('msgs')
])
prompt_template.invoke({
    'msgs': [
        HumanMessage(content='hi!')
    ]
})


In [ ]:
# 存储对话历史内容
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt_template = ChatPromptTemplate.from_messages([
    ('system','你是一个非常友好的AI助手'),
    MessagesPlaceholder(variable_name='history'),
    ('human','{question}')
])
prompt_template.invoke({
    'history': [
        ('human','5 + 2 = ?'),
        ('ai','5 + 2 = 7')
    ],
    'question':'结果再乘以4呢？'
})

In [ ]:
# 可复用模板库
from templates import PromptLibrary

messages = PromptLibrary.TRANSLATOR.format_messages(
   source_lang='英语',
    target_lang='中文',
    text='Hello World'
)
print(messages)
